In [ ]:
import os
import shutil
import random
import numpy as np
import rasterio
import torch
import subprocess
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, jaccard_score
from torch.utils.data import Dataset
from transformers import (
    SegformerImageProcessor,
    SegformerForSemanticSegmentation,
    TrainingArguments,
    Trainer
)

# --- Experiment Configuration ---
class Config:
    # Experiment Identifiers
    EXPERIMENT_NAME = "segformer_b0_turkey_wildfire"

    # 1. Dataset Paths (Prioritize Local, then Drive)
    # If you have the file locally, put it in the same folder as this script.
    LOCAL_ZIP_PATH = Path("./TurkeyWildfire.zip")
    DRIVE_ZIP_PATH = Path("/content/drive/MyDrive/Satellite/TurkeyWildfire.zip")

    EXTRACT_DIR = Path("./wildfire_data")

    # Data Processing
    # Logic preserved: Images are 16-bit, clipped to 5000
    CLIP_VALUE = 5000.0
    IMAGE_SIZE = (128, 128)
    TEST_SPLIT = 0.2

    # Model Architecture
    MODEL_CHECKPOINT = "nvidia/mit-b0"
    NUM_LABELS = 2
    ID2LABEL = {0: "Background", 1: "Wildfire"}
    LABEL2ID = {"Background": 0, "Wildfire": 1}

    # Training Hyperparameters
    BATCH_SIZE = 16
    LEARNING_RATE = 6e-5
    EPOCHS = 10
    SEED = 42
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    # Reproducibility
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

print(f"⚙️ Configuration loaded for: {Config.EXPERIMENT_NAME}")
print(f"🔧 Device: {Config.DEVICE}")

In [ ]:
class DataManager:
    """
    Handles data extraction from Local storage or Google Drive.
    Specific logic: Unzip > Unrar > Standardize Folder Names.
    """
    @staticmethod
    def _check_system_deps():
        """Checks for 'unrar' and installs it on Colab/Linux if missing."""
        if shutil.which("unrar") is None:
            print("⚠️ 'unrar' not found. Attempting to install...")
            if os.path.exists("/content"): # We are in Colab
                os.system("apt-get -q install unrar > /dev/null")
                print("✅ 'unrar' installed.")
            else:
                print("❌ 'unrar' is missing on your local machine!")
                print("   - Mac: brew install carlocab/personal/unrar")
                print("   - Windows: Install WinRAR and add to PATH")
                raise SystemError("Please install 'unrar' to extract this dataset.")

    @staticmethod
    def _get_zip_source():
        """Finds the dataset zip file."""
        # 1. Check Local
        if Config.LOCAL_ZIP_PATH.exists():
            print(f"✅ Found local dataset: {Config.LOCAL_ZIP_PATH}")
            return Config.LOCAL_ZIP_PATH

        # 2. Check Drive (if in Colab)
        if os.path.exists("/content"):
            if not Config.DRIVE_ZIP_PATH.exists():
                print("mount Google Drive...")
                from google.colab import drive
                drive.mount('/content/drive')

            if Config.DRIVE_ZIP_PATH.exists():
                print(f"✅ Found Drive dataset: {Config.DRIVE_ZIP_PATH}")
                return Config.DRIVE_ZIP_PATH

        raise FileNotFoundError("❌ TurkeyWildfire.zip not found! Please upload it or put it in Google Drive.")

    @classmethod
    def setup_dataset(cls):
        # 1. Check if data already exists (Skip if ready)
        if cls._is_data_ready():
            print(f"✅ Data already ready in {Config.EXTRACT_DIR}")
            return

        cls._check_system_deps()
        print("🔄 Initializing data setup...")

        # 2. Get Zip Source
        zip_source = cls._get_zip_source()

        # 3. Prepare Directories
        work_dir = Path("./temp_work")
        if work_dir.exists(): shutil.rmtree(work_dir)
        work_dir.mkdir(parents=True, exist_ok=True)
        Config.EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

        # 4. Extract Outer Zip
        print(f"📦 Unzipping {zip_source.name}...")
        # Use python's zipfile or system unzip depending on preference. System is often faster.
        os.system(f"unzip -q \"{zip_source}\" -d {work_dir}")

        # 5. Extract Inner RAR (Specific to TurkeyWildfire dataset)
        rar_file = next(work_dir.rglob("Dataset.rar"), None)
        if not rar_file:
            raise FileNotFoundError("❌ Dataset.rar not found inside zip.")

        print("📦 Extracting internal RAR...")
        # 'x' extract with full path, '-inul' disable all messages, '-o+' overwrite
        exit_code = os.system(f"unrar x -inul -o+ \"{rar_file}\" \"{work_dir}/\"")
        if exit_code != 0:
            raise RuntimeError("Unrar failed. Check if the RAR file is corrupt.")

        # 6. Standardize Folders (Handle spaces/underscores variations)
        print("📂 Standardizing folder structure...")
        cls._standardize_folders(work_dir)

        # Cleanup
        shutil.rmtree(work_dir)
        print(f"✅ Data setup complete. Saved to {Config.EXTRACT_DIR}")

    @classmethod
    def _standardize_folders(cls, search_dir):
        # Finds 'Train Images' or 'train_images' (case insensitive)
        img_src = next((p for p in search_dir.rglob("*") if p.name.lower() in ["train images", "train_images"] and p.is_dir()), None)
        msk_src = next((p for p in search_dir.rglob("*") if p.name.lower() in ["train masks", "train_masks"] and p.is_dir()), None)

        if img_src and msk_src:
            # Create final destination folders
            (Config.EXTRACT_DIR / "images").mkdir(parents=True, exist_ok=True)
            (Config.EXTRACT_DIR / "masks").mkdir(parents=True, exist_ok=True)

            # Move contents
            for f in img_src.glob("*"): shutil.move(str(f), str(Config.EXTRACT_DIR / "images"))
            for f in msk_src.glob("*"): shutil.move(str(f), str(Config.EXTRACT_DIR / "masks"))
        else:
            raise FileNotFoundError("Could not locate image/mask folders in extracted data.")

    @classmethod
    def _is_data_ready(cls):
        # Simple check: do we have images and masks folders?
        return (Config.EXTRACT_DIR / "images").exists() and (Config.EXTRACT_DIR / "masks").exists()

# Trigger Setup
DataManager.setup_dataset()

In [ ]:
class TurkeyWildfireDataset(Dataset):
    """
    Custom dataset for Turkey Wildfire data (16-bit TIFs).
    """
    def __init__(self, image_paths, mask_paths, processor, config):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.processor = processor
        self.config = config

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # 1. Load Image
        with rasterio.open(self.image_paths[idx]) as src:
            image = src.read() # shape: (Channels, Height, Width)

        # 2. Load Mask
        with rasterio.open(self.mask_paths[idx]) as src:
            mask = src.read(1) # shape: (Height, Width)

        # 3. Custom Normalization (Preserved logic from original)
        # Images are 16-bit, clipped to 5000, then normalized to [0, 1]
        image = np.clip(image, 0, self.config.CLIP_VALUE) / self.config.CLIP_VALUE

        # 4. Format Fix: Move axis for Processor (C,H,W -> H,W,C)
        # Transformers library expects channels last for input
        image = np.moveaxis(image, 0, -1).astype(np.float32)

        # 5. Processor
        encoded_inputs = self.processor(
            images=image,
            segmentation_maps=mask,
            return_tensors="pt",
            do_resize=False, # Data is pre-cropped/sized
            do_rescale=False,
            do_normalize=True
        )

        return {
            "pixel_values": encoded_inputs.pixel_values.squeeze(),
            "labels": encoded_inputs.labels.squeeze().long()
        }

def prepare_splits(config):
    """Synchronizes files and performs train/val split."""
    img_dir = config.EXTRACT_DIR / "images"
    msk_dir = config.EXTRACT_DIR / "masks"

    # Robust matching (case insensitive extension check)
    img_files = {p.stem: p for p in img_dir.glob("*") if p.suffix.lower() in ['.tif', '.tiff']}
    msk_files = {p.stem: p for p in msk_dir.glob("*") if p.suffix.lower() in ['.tif', '.tiff']}

    # Intersect to ensure we only use matched pairs
    common = sorted(set(img_files.keys()) & set(msk_files.keys()))
    print(f"📊 Found {len(common)} paired samples.")

    if len(common) == 0:
        raise ValueError("No matching image/mask pairs found! Check your data directory.")

    all_imgs = [img_files[k] for k in common]
    all_msks = [msk_files[k] for k in common]

    # Split
    return train_test_split(all_imgs, all_msks, test_size=config.TEST_SPLIT, random_state=config.SEED)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    # Convert logits to tensor
    logits_tensor = torch.from_numpy(logits)
    # labels are already numpy, but we might need tensor operations if resizing
    labels_tensor = torch.from_numpy(labels)

    # Upsample logits to label size if necessary (SegFormer outputs 1/4 resolution)
    if logits_tensor.shape[-2:] != labels_tensor.shape[-2:]:
        logits_tensor = torch.nn.functional.interpolate(
            logits_tensor,
            size=labels_tensor.shape[-2:],
            mode="bilinear",
            align_corners=False
        )

    # Get predictions
    pred_labels = logits_tensor.argmax(dim=1).numpy().flatten()
    true_labels = labels.flatten()

    # Metrics calculation
    iou = jaccard_score(true_labels, pred_labels, average='macro')
    acc = accuracy_score(true_labels, pred_labels)

    return {
        "iou": iou,
        "accuracy": acc
    }

In [ ]:
# 1. Prepare Data Splits
train_imgs, val_imgs, train_msks, val_msks = prepare_splits(Config)

# 2. Initialize Processor & Model
print(f"🚀 Loading model: {Config.MODEL_CHECKPOINT}")
processor = SegformerImageProcessor.from_pretrained(
    Config.MODEL_CHECKPOINT,
    do_resize=False,
    do_rescale=False,
    do_normalize=True
)

# Instantiate Datasets
train_ds = TurkeyWildfireDataset(train_imgs, train_msks, processor, Config)
val_ds = TurkeyWildfireDataset(val_imgs, val_msks, processor, Config)

print(f"📂 Train Size: {len(train_ds)} | Val Size: {len(val_ds)}")

model = SegformerForSemanticSegmentation.from_pretrained(
    Config.MODEL_CHECKPOINT,
    num_labels=Config.NUM_LABELS,
    id2label=Config.ID2LABEL,
    label2id=Config.LABEL2ID,
    ignore_mismatched_sizes=True
)

# 3. Define Training Arguments
args = TrainingArguments(
    output_dir=f"./results/{Config.EXPERIMENT_NAME}",
    learning_rate=Config.LEARNING_RATE,
    num_train_epochs=Config.EPOCHS,
    per_device_train_batch_size=Config.BATCH_SIZE,
    per_device_eval_batch_size=Config.BATCH_SIZE,
    save_strategy="epoch",
    evaluation_strategy="epoch",
    save_total_limit=2,
    logging_steps=50,
    fp16=(Config.DEVICE == "cuda"), # Use mixed precision if on GPU
    remove_unused_columns=False,
    report_to="none",
    dataloader_num_workers=2
)

# 4. Initialize Trainer
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

# 5. Start Training
print(f"🔥 Starting training on {Config.DEVICE}...")
trainer.train()

# 6. Save Final Model
save_path = f"./models/{Config.EXPERIMENT_NAME}"
trainer.save_model(save_path)
processor.save_pretrained(save_path)
print(f"🏆 Training finished. Model saved to {save_path}")